In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load dataset
df = pd.read_csv('C:/Users/HP/Downloads/Nupur Sarkar_CuvetteDS/Nupur Sarkar_Machinelearning/bestSelling_games.csv')  # replace with actual file path



In [15]:
df.size

35700

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380 entries, 0 to 2379
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   game_name            2380 non-null   object 
 1   reviews_like_rate    2380 non-null   int64  
 2   all_reviews_number   2380 non-null   int64  
 3   release_date         2380 non-null   object 
 4   developer            2380 non-null   object 
 5   user_defined_tags    2380 non-null   object 
 6   supported_os         2380 non-null   object 
 7   supported_languages  2380 non-null   object 
 8   price                2380 non-null   float64
 9   other_features       2380 non-null   object 
 10  age_restriction      2380 non-null   int64  
 11  rating               2380 non-null   float64
 12  difficulty           2380 non-null   int64  
 13  length               2380 non-null   int64  
 14  estimated_downloads  2380 non-null   int64  
dtypes: float64(2), int64(6), object(7)
mem

In [19]:
# 2. Data Cleaning
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month

# Handle missing values
df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0)
df['difficulty'] = df['difficulty'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unrated')

# Price Binning
df['price_bin'] = pd.cut(df['price'], bins=[-0.1, 0, 10, 30, np.inf],
                         labels=['Free', '<$10', '$10–30', '>$30'])

# 3. Feature Engineering
df['is_free'] = df['price'] == 0
df['review_score'] = df['reviews_like_rate'] * df['all_reviews_number']
df['review_volume_bucket'] = pd.qcut(df['all_reviews_number'], 3, labels=['Low', 'Medium', 'High'])

# Count-based features
df['tag_count'] = df['user_defined_tags'].fillna('').apply(lambda x: len(str(x).split(',')))
df['lang_count'] = df['supported_languages'].fillna('').apply(lambda x: len(str(x).split(',')))
df['os_count'] = df['supported_os'].fillna('').apply(lambda x: len(str(x).split(',')))

# Binarize multi-label fields
mlb = MultiLabelBinarizer()

def binarize_column(df, column):
    temp = df[column].fillna('').apply(lambda x: [i.strip() for i in str(x).split(',')])
    return pd.DataFrame(mlb.fit_transform(temp), columns=[f"{column}_{cls}" for cls in mlb.classes_])

df_tags = binarize_column(df, 'user_defined_tags')
df_os = binarize_column(df, 'supported_os')
df_features = binarize_column(df, 'other_features')

# Combine all features
df_ml = pd.concat([
    df[['estimated_downloads', 'review_score', 'tag_count', 'lang_count', 'os_count', 'is_free']],
    df_tags, df_os, df_features
], axis=1)

# Target Variable
df_ml['target'] = (df['estimated_downloads'] > df['estimated_downloads'].median()).astype(int)

# 4. Train-Test Split
X = df_ml.drop(columns='target')
y = df_ml['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Modeling
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}

# 6. Evaluation
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"Precision: {precision_score(y_test, y_pred):.2f}")
    print(f"Recall: {recall_score(y_test, y_pred):.2f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.2f}")
    print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.2f}\n")


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression ===
Accuracy: 0.99
Precision: 0.99
Recall: 1.00
F1-Score: 0.99
AUC-ROC: 1.00

=== Random Forest ===
Accuracy: 1.00
Precision: 1.00
Recall: 1.00
F1-Score: 1.00
AUC-ROC: 1.00

=== Gradient Boosting ===
Accuracy: 1.00
Precision: 1.00
Recall: 1.00
F1-Score: 1.00
AUC-ROC: 1.00

